In [1]:
import os 
from scipy import sparse

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objs as go

In [2]:
os.getcwd()

'/home/czy/KT/BRIKT_mine/dataset/assist09'

In [3]:
class DataProcess():
    def __init__(self, data_folder='assist09', file_name='skill_builder_data_corrected_collapsed.csv', min_inter_num=3):
        print("Process Dataset %s" % data_folder)
        self.min_inter_num = min_inter_num
        self.data_folder = data_folder
        self.file_name = file_name

    # delete empty skill_id, empty skill_name，生成assist09_processed.csv文件
    def process_csv(self):
        """
            pre-process original csv file for assist dataset
        """
        # read csv file
        data_path = self.file_name
        df = pd.read_csv(data_path, low_memory=False, encoding="ISO-8859-1")
        print('original records number %d' % len(df)) # 346860
        # todo we remove records without skills and scaffolding problems
        # delete empty skill_id, empty skill_name
        df = df.dropna(subset=['skill_id', 'skill_name'])
        #
        print(len(df)) # 274590
        df = df[~df['skill_id'].isin(['noskill'])]
        #
        print(len(df)) # 274590
        print('After removing empty skill_id and empty skill_name, records number %d' % len(df)) # 274590
        df.to_csv('%s_processed.csv'%self.data_folder, index=None)
    
    # 生成new_pro_id和new_skill_id的图pro_skill_sparse.npz
    # 生成包含多个特征数组[[[ms,1.0,0.0, 0.0, 0.0, 0.0,p], ...],...]的文件pro_feat.npz
    def pro_skill_graph(self):
        df = pd.read_csv('%s_processed.csv'%self.data_folder,low_memory=False, encoding="ISO-8859-1")
        problems = df['problem_id'].unique() 
        pro_id_dict = dict(zip(problems, range(len(problems))))
        print('problem number %d' % len(problems)) # 16891
        # print('打印problem的字典{problem_id: index}')
        # print(pro_id_dict) # {51424: 0, 51435: 1, 51444: 2, ..., 128416: 16889, 128410: 16890}

        skills = df['skill_name'].unique()
        skill_id_dict = dict(zip(skills, range(len(skills))))
        print('skills number %d' % len(skills)) # 101
        # print('打印skill的字典{skill_name: index}')
        # print(skill_id_dict) # {'Box and Whisker': 0, 'Circle Graph': 1, ..., 'Solving Systems of Linear Equations by Graphing': 100}

        pro_type = df['answer_type'].unique()
        pro_type_dict = dict(zip(pro_type, range(len(pro_type))))
        # print('打印problem type的字典{answer_type: index}')
        print('problem type: ', pro_type_dict) # {'algebra': 0, 'fill_in_1': 1, 'choose_1': 2, 'open_response': 3, 'choose_n': 4}

        pro_feat = []
        pro_skill_adj = []
        # skill_id_dict, skill_cnt = {}, 0
        for pro_id in range(len(problems)):            
            tmp_df = df[df['problem_id']==problems[pro_id]]
            tmp_df_0 = tmp_df.iloc[0]
            # print(tmp_df)
            # print(tmp_df_0)
            # print(tmp_df_0['answer_type']) # algebra
            # pro_feature: [ms_of_response, answer_type, mean_correct_num]
            ms = tmp_df['ms_first_response'].abs().mean()
            # print(ms) # 2094.0
            p = tmp_df['correct'].mean()
            # print(p) # 1.0
            pro_type_id = pro_type_dict[tmp_df_0['answer_type']]
            # print(pro_type_id) # 0
            tmp_pro_feat = [0.] * (len(pro_type_dict)+2)
            # print(tmp_pro_feat) # [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
            tmp_pro_feat[0] = ms
            tmp_pro_feat[pro_type_id+1] = 1.
            tmp_pro_feat[-1] = p
            pro_feat.append(tmp_pro_feat)
            # print(pro_feat) # [[28844.5, 1.0, 0.0, 0.0, 0.0, 0.0, 0.7647058823529411], ... ]里面每个元组表示[[ms,1.0,0.0, 0.0, 0.0, 0.0,p], ...]
            # print(tmp_df_0['skill_name']) # Solving Systems of Linear Equations by Graphing
            # print(skill_id_dict[tmp_df_0['skill_name']]) # 100

            # pro_skill_adj.append([pro_id, skill_id_dict[tmp_df_0['skill_name']], 1])
            pro_skill_adj.append([pro_id, skill_id_dict[tmp_df_0['skill_name']]]) # 自己改的，不保存权重

            
        # print(pro_skill_adj) # 问题-知识点的图[[new_pro_id, new_skill_id, 1], ...]

        pro_skill_adj = np.array(pro_skill_adj).astype(np.int32)

        pro_feat = np.array(pro_feat).astype(np.float32)
        # 对ms这一列的值做归一化？也不是归一化，反正做了处理
        pro_feat[:, 0] = (pro_feat[:, 0] - np.min(pro_feat[:, 0])) / (np.max(pro_feat[:, 0])-np.min(pro_feat[:, 0]))
        # print(pro_feat)
        # print(pro_feat.shape) # (16891, 7)

        pro_num = np.max(pro_skill_adj[:, 0]) + 1
        skill_num = np.max(pro_skill_adj[:, 1]) + 1
        # 打印problem的个数为16891，skill的个数为101
        print('problem number %d, skill number %d' % (pro_num, skill_num))

        # save pro-skill-graph in sparse matrix form 坐标格式的稀疏矩阵，可查阅例子https://blog.csdn.net/lonelykid96/article/details/102725668
        pro_skill_sparse = sparse.coo_matrix((pro_skill_adj[:, 2].astype(np.float32), (pro_skill_adj[:, 0], pro_skill_adj[:, 1])), shape=(pro_num, skill_num))
        sparse.save_npz('pro_skill_sparse.npz', pro_skill_sparse)

        # save pro-id-dict, skill-id-dict, pro_type_dict
        self.save_dict(pro_id_dict, 'pro_id_dict.txt')
        self.save_dict(skill_id_dict, 'skill_id_dict.txt')
        self.save_dict(pro_type_dict, 'pro_type_dict.txt') # 自己加的

        # save pro_feat_arr 
        # np.savez将多个数组保存到一个文件中的话，可以使用numpy.savez函数。savez函数的第一个参数是文件名，其后的参数都是需要保存的数组
        # np.savez('pro_feat.npz', pro_feat=pro_feat)

    # 存储文件的函数
    def save_dict(self, dict_name, file_name):
        f = open(file_name, 'w')
        f.write(str(dict_name))
        f.close
    # 存储文件的函数
    def write_txt(self, file, data):
        with open(file, 'w') as f:
            for dd in data:
                for d in dd:
                    f.write(str(d)+'\n')
    
    # generate user interaction sequence
    # and write to data.txt
    # 4151个user的记录，每个user的记录包括：记录个数，skill_name, problem_id, correct,生成data.txt
    def generate_user_sequence(self, seq_file):
        df = pd.read_csv('%s_processed.csv'%self.data_folder, low_memory=False, encoding="ISO-8859-1")
        # print(len(df)) # 274590
        ui_df = df.groupby(['user_id'], as_index=True) 
        # print(len(ui_df)) # 4151
        print('user number %d' % len(ui_df)) # 4151,表示4151个studennts

        user_inters = []
        cnt = 0
        for ui in ui_df:
            tmp_user, tmp_inter = ui[0], ui[1]
        # print(tmp_user) # 96299，某个user_id
        # print(tmp_inter) # 367, user_id为96299的367条记录
            tmp_problems = list(tmp_inter['problem_id'])
            tmp_skills = list(tmp_inter['skill_name'])
            tmp_ans = list(tmp_inter['correct'])
            user_inters.append([[len(tmp_inter)], tmp_skills, tmp_problems, tmp_ans])
        # print(len(user_inters)) # 4151, user个数
        # print(len(tmp_inter))
        # print(user_inters[0])
        # write_file = os.path.join(self.data_folder, seq_file)
        self.write_txt(seq_file, user_inters)
        


    def read_user_sequence(self, filename, max_len=200, min_len=3, shuffle_flag=True):
        with open(filename, 'r') as f:
            lines = f.readlines() # 按行读取data.txt文件
            # print(len(lines)) # 16604, 16604=4151×4,4151表示user个数，4表示：记录个数，skill_name, problem_id, correct
        with open('skill_id_dict.txt', 'r') as f:
            skill_id_dict = eval(f.read()) 
            # print(skill_id_dict) # {'Box and Whisker': 0, 'Circle Graph': 1, 'Histogram as Table or Graph': 2, ...}
        with open('pro_id_dict.txt', 'r') as f:
            pro_id_dict = eval(f.read())
            # print(pro_id_dict) # {51424: 0, 51435: 1, 51444: 2, 51395: 3, 51481: 4, ...}
        with open('../dataset/assist09/pro_type_dict.txt', 'r') as f:
            pro_type_dict = eval(f.read) # 自己加的
           
        y, skill, problem, real_len = [], [], [], []
        skill_num, pro_num = len(skill_id_dict), len(pro_id_dict)
        print('skill num, pro num, ', skill_num, pro_num) # skill num, pro num,  101 16891
        index = 0
        while index < len(lines):
            num = eval(lines[index])[0]
            tmp_skills = eval(lines[index+1])[:max_len]
            # print(tmp_skills) # skill_name
            tmp_skills = [skill_id_dict[ele]+1 for ele in tmp_skills]
            # print(tmp_skills) # new_skill_name # for assist09
            # tmp_skills = [ele+1 for ele in tmp_skills]  # for assist12
            tmp_pro = eval(lines[index+2])[:max_len]
            # print(tmp_pro) # problem_id
            tmp_pro = [pro_id_dict[ele]+1 for ele in tmp_pro]
            # print(tmp_pro)
            tmp_ans = eval(lines[index+3])[:max_len]
            # print(tmp_ans) # 每个user的记录中的answer

            if num>=min_len:
                tmp_real_len = len(tmp_skills)
                # Completion sequence
                tmp_ans += [-1]*(max_len-tmp_real_len)
                # print(tmp_ans) # 每个user的记录补成长度为200的
                tmp_skills += [0]*(max_len-tmp_real_len)
                # print(tmp_skills)
                tmp_pro += [0]*(max_len-tmp_real_len)
                y.append(tmp_ans)
                # print(y)
                skill.append(tmp_skills)
                problem.append(tmp_pro)
                real_len.append(tmp_real_len)
                
            index += 4
            # print(num) # 19 17 6 6 268 ... 367

        y = np.array(y).astype(np.float32)
        skill = np.array(skill).astype(np.int32)
        problem = np.array(problem).astype(np.int32)
        real_len = np.array(real_len).astype(np.int32)

        print(skill.shape, problem.shape, y.shape, real_len.shape) # (3831, 200) (3831, 200) (3831, 200) (3831,)
        print(np.max(y), np.min(y)) # 1.0 -1.0
        print(np.max(real_len), np.min(real_len)) # 200 3
        print(np.max(skill), np.min(skill)) # 101 0
        print(np.max(problem), np.min(problem)) # 16890 0

        np.savez("%s.npz"%self.data_folder, problem=problem, y=y, skill=skill, real_len=real_len, skill_num=skill_num, problem_num=pro_num)



In [4]:
data_folder = 'assist09'
min_inter_num = 3
file_name='skill_builder_data_corrected_collapsed.csv'

In [5]:
DP = DataProcess(data_folder, file_name, min_inter_num)

Process Dataset assist09


数据集skill_builder_data_corrected_collapsed.csv文件 的 original records number 346860

After removing empty skill_id, records number 274590 存储在assist09_processed.csv当中

### 对原始数据集skill_builder_data_corrected_collapsed.csv文件做处理；

 删除含有空的 skill_name的记录，生成assist09_processed.csv文件

 原始数据集skill_builder_data_corrected_collapsed.csv文件含有346860条记录

 assist09_processed.csv文件含有274590条记录，包含4151个user, 16891个problem_id, 101个skiill_name, 5种problem_type

In [6]:
# DP.process_csv()

problem个数16891，通过表中problem_id属性计算的

skill个数101，通过表中skill_name属性计算的

problem_type个数为5，通过表中answer_type属性计算的

## 生成pro_skill_sparse.npz，pro_id_dict.txt，skill_id_dict.txt，pro_feat.npz等文件

In [7]:
# DP.pro_skill_graph()

## 生成data.txt文件

In [8]:
# DP.generate_user_sequence('data.txt')

In [9]:
# DP.read_user_sequence('data.txt')

# edge
## 生成edge_list.dat文件

#### 注意这个数据集用的skill_name而不是skill_id

In [10]:
df = pd.read_csv('assist09_processed.csv', encoding="ISO-8859-1", low_memory=False)
# df.columns # ['Unnamed: 0', 'order_id', 'assignment_id', 'user_id', 'assistment_id',..., 'opportunity_original']
# df.shape # (274590, 31)
problems = df['problem_id'].unique() # 获取去重的问题序列
pro_id_dict = dict(zip(problems, range(len(problems)))) # 问题id：index
print('problem number %d' % len(problems)) # 16891

# skill_id_dict, skill_cnt = {}, 0
skills = df['skill_name'].unique() # 注意设个数据集用的skill_name而不是skill_id
skill_id_dict = dict(zip(skills, range(len(skills))))
print('skill number %d' % len(skills)) # 101

# 构造图：q_id, s_id, weight
# 先得到独一无二的q_id
# 根据tmp_df_0得到此时q_id对应的skill_name
# 再根据skill_name从字典skill_id_dict中查到对应的s_id
# 权重这里都设置为1
file = open(os.path.join(os.getcwd(), 'edge_list.dat'), 'w')
file.write("{}\t{}\t{}\n".format('qid', 'sid', 'weight'))
for pro_id in range(len(problems)): 
    tmp_df = df[df['problem_id']==problems[pro_id]]
    tmp_df_0 = tmp_df.iloc[0]  # problem_id
    # build problem-skill bipartite
    file.write("{}\t{}\t{}\n".format(pro_id, skill_id_dict[tmp_df_0['skill_name']], 1)) # 16871条记录

# beacuse the padding of q and s is num_q+1 and num_s+1
file.write("{}\t{}\t{}\n".format(len(problems), len(skill_id_dict), 1)) # 这里变成了16892条
file.close()

problem number 16891
skill number 101


#### 构建得到的图有16891个q，101个s，q和s互连的边有16871条，可以在edge_list.dat文件中查看